In [ ]:
import weaviate
from weaviate.classes.config import Configure, DataType, Multi2VecField, Property


In [ ]:
client = weaviate.connect_to_local(
    host="172.17.0.5",  
    port=8080,
    grpc_port=50051,
 
)
print(client.is_ready())



In [ ]:
from weaviate.classes.config import Property, DataType, Configure, Multi2VecField

questions = client.collections.create(
    "unified_embedding",
    properties=[
        Property(
            name="type",
            data_type=DataType.TEXT,
            skip_vectorization=True
        ),
        Property(
            name="page",
            data_type=DataType.INT,
            skip_vectorization=True
        ),
        Property(
            name="description",
            data_type=DataType.TEXT,
            skip_vectorization=False
        ),
        Property(
            name="text",
            data_type=DataType.TEXT,
            skip_vectorization=False
        ),
        Property(
            name="trace",
            data_type=DataType.TEXT,
            skip_vectorization=True  # Changed to True
        ),
        Property(
            name="filename",
            data_type=DataType.TEXT,
            skip_vectorization=True  # Changed to True
        ),
        Property(
            name="image",
            data_type=DataType.BLOB,
            skip_vectorization=False
        ),
    ],
    vectorizer_config=[
        Configure.Vectorizer.multi2vec_clip(
            name="combined_vector",
            image_fields=[
                Multi2VecField(name="image", weight=0.5)
            ],
            text_fields=[
                Multi2VecField(name="description", weight=0.25),  # Adjusted weights
                Multi2VecField(name="text", weight=0.25)  # Now sum to 1.0
            ]
        )
    ]
)

In [ ]:
import csv

with open("example.csv", "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    source_objects = list(reader)


In [ ]:
collection = client.collections.use("DemoCollection")

with collection.batch.fixed_size(batch_size=200) as batch:
    for src_obj in source_objects:
        poster_b64 = url_to_base64(src_obj["poster_path"])
        weaviate_obj = {
            "title": src_obj["title"],
            "poster": poster_b64  # Add the image in base64 encoding
        }

        # The model provider integration will automatically vectorize the object
        batch.add_object(
            properties=weaviate_obj,
            # vector=vector  # Optionally provide a pre-obtained vector
        )

In [ ]:
from PIL import Image
import io
import base64
collection = client.collections.use("DemoCollection")

response = collection.query.near_text(
    query="diagram",  # The model provider integration will automatically vectorize the query
    limit=2,
    return_properties=["title", "poster"]  # Optionally return specific properties
)

for obj in response.objects:
    print(obj.properties["title"])
    poster_b64 = obj.properties["poster"]
    poster_bytes = base64.b64decode(poster_b64)
    image = Image.open(io.BytesIO(poster_bytes))
    image.show()



In [ ]:
client.close()